<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/08-data_cleaning_preprocessing/08-data_cleaning_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 8: Data Cleaning and Preprocessing

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0), especially the chapter on *Data Cleaning and Preprocessing*
- [pandas documentation](https://pandas.pydata.org/docs/), in particular the guides on [missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) and [merging](https://pandas.pydata.org/docs/user_guide/merging.html)
- Module 7, whose findings memo is this module's starting point

```
ASK  ->  GET  ->  [ CLEAN  <->  EXPLORE ]  ->  MODEL  ->  COMMUNICATE
```

Notice the two-way arrow. In Module 7 we *explored* the taxi data and produced an inventory of problems. Today we act on that inventory, and the acting will send us back to explore more than once. Cleaning and exploring are two hands of the same craft.

## Learning objectives

1. Explain why cleaning requires documented judgment calls rather than default settings.
2. Recognize how missing values are represented in Python and detect them, including disguised ones.
3. Choose and implement deletion and imputation strategies, and explain their trade-offs.
4. Repair wrongly typed or inconsistently formatted columns.
5. Detect and remove duplicate records.
6. Combine data from multiple tables with `pd.merge()`.
7. Handle outliers with documented, purpose-driven decisions.
8. Assemble a reproducible cleaning script and defend every choice in it.

Our dataset is an old friend: the **6,433 NYC taxi rides** from Module 7. By the end of this module we will have turned it into `taxis_clean`, a documented, analysis-ready table, and you will be able to defend every row we changed.

> **Instructor note (pacing).** Again roughly three class periods, with surplus:
> - **Day 1:** Sections 1 and 2 (decisions, missing data, the imputation experiment).
> - **Day 2:** Sections 3 to 5 (types and formats, duplicates, merging).
> - **Day 3:** Sections 6 and 7 (outlier decisions, the capstone cleaning script).
>
> Cut candidates if pressed: the KNN k-comparison (Activity 4), the outer-join Staten Island reveal (5.3), and the log-scale histogram (6.1). The Section 7 capstone is the point of the module; protect it.

---
## 1. From Detection to Decision
---

Module 7 ended with a findings memo. The relevant lines, our work orders for today:

1. **44 rides** have no recorded `payment`.
2. **51 rides** show zero distance but a positive fare (up to 120 dollars).
3. **96 rides** report zero passengers.
4. **6 rides** end at or before the moment they begin.
5. The location columns have 26 to 45 missing values each.
6. Cash tips are structurally recorded as zero; no cleaning can recover them.

In Module 7 the question was *"what is wrong?"* Today the question changes: *"what do we do about it, and how do we justify it?"* That word **justify** is the theme of the module. Cleaning is not a chore you run; it is a series of small judgment calls, and each call can change your conclusions.

### 1.1 Three rules before we touch anything

**Rule 1: never edit the raw data.** The original file stays untouched, forever. All changes happen in code, on copies. If a decision turns out to be wrong next week, we rerun the script with one line changed.

**Rule 2: every change gets a written reason.** Six months from now, someone (probably you) will ask "why are there only 6,382 rows in this table?" The answer should be written down next to the code that dropped them.

**Rule 3: there is no such thing as clean, only clean *enough for a purpose*.** A table prepared for a fare model and a table prepared for a tipping study will make *different* choices about the very same rows. You will see this concretely in the capstone.

These rules are why data cleaning lives in a script instead of in hand-edits to a spreadsheet: a script is reproducible (anyone can rerun it), auditable (every change is visible), and reversible (delete a line, rerun).

In [ ]:
# RUN-TOGETHER
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
# RUN-TOGETHER
# Reload the raw data and replay our Module 7 repairs, in one cell.
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv'
taxis = pd.read_csv(url)

taxis['pickup'] = pd.to_datetime(taxis['pickup'])
taxis['dropoff'] = pd.to_datetime(taxis['dropoff'])
taxis['duration_min'] = (taxis['dropoff'] - taxis['pickup']).dt.total_seconds() / 60
taxis['pickup_hour'] = taxis['pickup'].dt.hour
taxis['pickup_day'] = taxis['pickup'].dt.day_name()

print(taxis.shape)

Pause on what just happened: five lines of code re-created *everything* we built across three class periods. That is Rule 1 in action. The raw file on the server is untouched; our repaired version exists only as the output of a script we can rerun, share, and improve.

### Activity 1 (Think-Pair-Share)

For each of the four headline problems (missing payments, zero-distance rides with fares, zero-passenger rides, impossible durations), decide with a partner which response fits best:

- **Drop** the affected rows,
- **Fix** the values (replace them with something better), or
- **Keep** them, flagged and documented.

Then the twist: does any of your answers change if the goal is **(i)** a model that predicts fares versus **(ii)** a study of tipping behavior? Be ready to defend one answer out loud.

> **Instructor note (Activity 1, about 6 minutes).** There is no answer key, which is the point; students should feel the decisions being *decisions*. Positions worth surfacing: zero-distance rides are toxic for a fare model but harmless for a tipping study restricted to card payments; missing payment is fatal for the tipping study but nearly irrelevant for the fare model; zero passengers might be a recording default rather than an empty cab, which suggests "fix to missing" rather than "drop". Collect the proposals on the board; the capstone in Section 7 will implement one defensible set and Activity 10 will implement a different one.

---
## 2. Missing Data
---

Missing data is the most common cleaning problem, and Python has accumulated several ways to represent "nothing here":

- `None`: Python's general-purpose "no value" object.
- `np.nan`: "Not a Number", the missing marker used in numeric computing. It is technically a *float*.
- `pd.NA`: pandas' newer, type-preserving missing marker.

When `read_csv()` meets an empty cell, it fills in `NaN`. You mostly do not choose among these representations; you just need to detect them reliably, and there is one genuine trap on the way.

In [ ]:
# RUN-TOGETHER
# The trap: NaN is not equal to anything, INCLUDING ITSELF.
print(np.nan == np.nan)

# So we never hunt for missing values with ==. We use isna().
s = pd.Series([4.5, np.nan, 7.0, None])
print(s.isna())
print('missing count:', s.isna().sum())

`np.nan == np.nan` is `False`. This is not a pandas bug; it is a rule from the international floating-point standard. The practical consequence is a habit: **missing values are found with `isna()` (or its mirror `notna()`), never with `==`**.

One more side effect worth knowing: because `np.nan` is a float, an integer column that gains even one missing value silently becomes a float column. If you ever see a count column holding values like `2.0`, missing data is usually the culprit. Remember this; it returns in the capstone.

### 2.1 Disguised missing values

`isna()` only finds *honest* missing values. Real datasets are full of dishonest ones: empty strings, `'unknown'`, `'N/A'` typed by a human, `-999`, or a default `0` that a device writes when it has nothing to report. To pandas these are perfectly valid values; only *meaning* makes them missing.

In [ ]:
# RUN-TOGETHER
# A tiny survey table with honest AND disguised missing values.
survey = pd.DataFrame({
    'name':   ['Alice', 'Bob', '', 'Diane'],
    'age':    [25, np.nan, 30, 28],
    'city':   ['La Crosse', 'unknown', 'Madison', ''],
})

print(survey)
print()
print('isna() finds:')
print(survey.isna().sum())

`isna()` reports one missing value in the whole table, but look at it: a blank name, an `'unknown'` city, a blank city. Three more holes, all invisible to `isna()`. Finding them takes comparisons and judgment.

In [ ]:
# RUN-TOGETHER
# eq() marks cells equal to a value; isin() checks against a list of suspects.
print(survey.eq('').sum())
print()
print(survey['city'].isin(['', 'unknown', 'N/A']).sum(), 'suspicious city entries')

Now back to the taxis, because we have been sitting on a suspected disguise since Module 7: **96 rides with `passengers == 0`.** An empty taxi completing a paid trip is hard to believe; a driver skipping the passenger-count button is easy to believe. Under that reading, those zeros are not data, they are *disguised missing values*, and the honest repair is to make them officially missing. We will do exactly that in the capstone.

### Activity 2 (short code, about 5 minutes)

Here is a fragment of a (fictional) driver sign-up sheet. Count its problems:

a. How many *honest* missing values does `isna()` find in each column?

b. How many entries of `license_id` are disguised missing values? Decide first which values count as disguises, then count them with `isin()`.

In [ ]:
# Activity 2
signups = pd.DataFrame({
    'driver':     ['Ford', 'Weber', 'Chen', 'Ortiz', 'Patel'],
    'shift':      ['night', np.nan, 'day', 'day', np.nan],
    'license_id': ['T-4471', 'missing', 'T-5502', '', 'n/a'],
})

# a. honest missing values per column
print(signups.____().sum())

# b. disguised missing values in license_id
suspects = [____, ____, ____]
print('disguised:', signups['license_id'].isin(suspects).____())

> **Instructor note (Activity 2).** (a) `shift` has 2 honest missing values, the others 0. (b) With suspects `['missing', '', 'n/a']` the count is 3. Worth asking the room: is `'n/a'` always a disguise? (No: in some datasets it means "not applicable", which is information, not absence. Meaning decides, not the string.)

### 2.2 Is the missingness random?

Before choosing a remedy, ask the diagnostic question from Module 7: **is the missingness random, or systematic?** Statisticians distinguish three flavors, which we will use informally:

- **Missing completely at random:** the holes have nothing to do with anything (a clerk spilled coffee on random pages). Dropping such rows loses data but adds no bias.
- **Missing at random given other columns:** the holes are predictable from *other observed* values (one cab company's meters fail more often). Trouble, but manageable if you know the pattern.
- **Missing not at random:** the holes depend on the *missing value itself* (high earners decline to report income). The dangerous kind.

We cannot always tell which flavor we have, but we can *look*. Let us interrogate our 44 missing payments: do those rides differ from the rest?

In [ ]:
# RUN-TOGETHER
missing_pay = taxis[taxis['payment'].isna()]

print('fare, missing-payment rides:', missing_pay['fare'].mean().round(2))
print('fare, all other rides:      ', taxis.loc[taxis['payment'].notna(), 'fare'].mean().round(2))
print()
print('tip summary for the 44 missing-payment rides:')
print(missing_pay['tip'].describe().round(2))

Fares look ordinary: 11.99 versus 13.10 dollars on average. But read the tip summary: **all 44 rides record a tip of exactly zero.** Minimum zero, maximum zero, standard deviation zero.

Where have we seen unanimous zero tips before? Cash rides. These 44 rides *behave exactly like cash rides*, which suggests the missingness is systematic: whatever prevented the payment type from being recorded also meant no card tip was recorded. This is not proof, but it is a strong hint, and it has a practical consequence: for any tipping analysis, treating these rides as "unknown, probably cash-like" is far more honest than pretending they are a random sample of all rides.

Thirty seconds of `describe()` on a suspicious subset. That is the whole technique, and it changed our understanding of those rows.

### 2.3 Remedy one: deletion

The bluntest remedy is dropping rows (or columns). The tool is `dropna()`, and its defaults deserve suspicion.

In [ ]:
# RUN-TOGETHER
print('original shape:              ', taxis.shape)
print('dropna() default:            ', taxis.dropna().shape)
print('dropna(subset=[payment]):    ', taxis.dropna(subset=['payment']).shape)

The default `dropna()` removes every row with *any* missing value: 92 rows gone, including rides whose only sin is a missing dropoff zone we may never need. The `subset` argument surgically drops rows missing a specific column: 44 rows, exactly the ones a payment-based analysis cannot use.

The vocabulary, with the trade-offs:

- **Listwise deletion** (`dropna()` with no arguments): simple, but discards partially useful rows and, if the missingness is systematic, *biases what remains*. Deleting our 44 cash-like rides quietly shifts the payment mix.
- **Targeted deletion** (`dropna(subset=...)`): drops only rows that are unusable *for your analysis*. Almost always the better form of deletion.
- **Column deletion** (`taxis.drop(columns=...)`): when a column is so hole-riddled it is beyond saving. None of ours is close.

Notice that every bullet ends in a judgment call, made *per analysis*. Rule 3 again.

### 2.4 Remedy two: imputation

**Definition.** *Imputation* is filling missing entries with estimated, plausible values, so that the rest of the row can still participate in the analysis.

For a **categorical** column the standard menu is: fill with the most common category (the *mode*), or fill with an explicit label like `'unknown'`. For our missing payments, think about what each would do:

In [ ]:
# RUN-TOGETHER
print('mode of payment:', taxis['payment'].mode()[0])

# Option A: fill with the mode          -> those 44 rides become 'credit card'
# Option B: fill with an honest label   -> those 44 rides become 'unknown'
option_b = taxis['payment'].fillna('unknown')
print()
print(option_b.value_counts())

Option A would stamp `'credit card'` onto 44 rides that we *just showed* behave like cash rides. The fill would be tidy, confident, and wrong, and no future user of the data could tell. Option B keeps the uncertainty visible. For categorical data, an honest `'unknown'` frequently beats a confident guess, precisely because it does not destroy the record of our ignorance.

For **numerical** columns the menu is richer, and we can do something better than argue about it: we can run an experiment where we *know the right answers*.

### 2.5 The imputation experiment

Our taxi table has no missing numeric values, which is an opportunity: we will **manufacture** some. We copy the data, secretly delete 300 `distance` values, impute them by various methods, and grade each method against the truth we hid.

Our score will be the **mean absolute error (MAE)**: the average size of the mistake, in miles. Lower is better.

In [ ]:
# RUN-TOGETHER
# Hide 300 distance values (but keep the answer key).
rng = np.random.default_rng(210)
hidden = rng.choice(taxis.index, size=300, replace=False)

answer_key = taxis.loc[hidden, 'distance'].copy()

experiment = taxis.copy()
experiment.loc[hidden, 'distance'] = np.nan
print('missing distances created:', experiment['distance'].isna().sum())

In [ ]:
# RUN-TOGETHER
# Method 1 and 2: fill every hole with the column mean, or the column median.
mean_filled = experiment['distance'].fillna(experiment['distance'].mean())
median_filled = experiment['distance'].fillna(experiment['distance'].median())

mae_mean = (mean_filled[hidden] - answer_key).abs().mean()
mae_median = (median_filled[hidden] - answer_key).abs().mean()
print('MAE, mean imputation:  ', round(mae_mean, 2), 'miles')
print('MAE, median imputation:', round(mae_median, 2), 'miles')

Mean imputation misses by 2.28 miles on average; the median does a bit better at 1.90 (less distracted by the long-trip tail, the same skew story as always). But step back: the *typical ride is only 1.6 miles long*. Both methods are guessing one number for every hole, so both are off by more than an entire typical trip. Constant fills are weak medicine.

Can we be smarter? Yes, and Module 7 already told us how: **distance and fare have a correlation of 0.92.** If a ride's distance is missing but its fare is 40 dollars, we should not guess "3 miles, the average". We should guess "whatever distance rides with 40-dollar fares usually have".

That is exactly the idea of **k-nearest-neighbors (KNN) imputation**: to fill a hole, find the k rows most similar on the columns we *do* have, and average their values.

In [ ]:
# RUN-TOGETHER
from sklearn.impute import KNNImputer

# Impute distance using fare as the similarity signal, k = 5 neighbors.
imputer = KNNImputer(n_neighbors=5)
filled = imputer.fit_transform(experiment[['distance', 'fare']])
knn_result = pd.Series(filled[:, 0], index=experiment.index)

mae_knn = (knn_result[hidden] - answer_key).abs().mean()
print('MAE, KNN using fare:', round(mae_knn, 2), 'miles')

In [ ]:
# RUN-TOGETHER
# Give KNN more signals: fare, duration, and tolls.
imputer = KNNImputer(n_neighbors=5)
filled = imputer.fit_transform(experiment[['distance', 'fare', 'duration_min', 'tolls']])
knn_rich = pd.Series(filled[:, 0], index=experiment.index)

mae_knn_rich = (knn_rich[hidden] - answer_key).abs().mean()
print('MAE, KNN using fare + duration + tolls:', round(mae_knn_rich, 2), 'miles')

The scoreboard: mean 2.28, median 1.90, KNN with fare 0.66, KNN with three signals **0.28 miles**. An eight-fold improvement over the naive fill, and every bit of it came from *relationships we discovered during EDA*. Good imputation is not a fancier formula; it is applied knowledge of the dataset. Modules 7 and 8 are one project.

Three caveats before imputation goes to your head:

- An imputed value is a *guess wearing the costume of data*. Always record how many values were imputed, and how.
- Imputation can artificially strengthen relationships (we filled distance *using* fare, so distance-fare correlation gets a free boost). Anyone modeling that pair afterward deserves to know.
- Never impute the variable that is the subject of your study. Filling holes in the thing you are trying to understand is manufacturing evidence.

### Activity 3 (short code, about 8 minutes)

The choice of k matters. Rerun the fare-only KNN imputation with **k = 1** and then **k = 1000**, computing the MAE for each. Before running, predict with a partner: will a single nearest neighbor beat k = 5? Will a thousand?

In [ ]:
# Activity 3
for k in [1, 5, 1000]:
    imputer = KNNImputer(n_neighbors=____)
    filled = imputer.fit_transform(experiment[['distance', 'fare']])
    result = pd.Series(filled[:, 0], index=experiment.index)
    mae = (result[____] - answer_key).abs().____()
    print('k =', k, ' MAE:', round(mae, 2))

> **Instructor note (Activity 3).** Approximate results with this seed: k = 1 gives 0.77, k = 5 gives 0.66, k = 1000 gives 0.79 (and k = 100, if someone tries it, about 0.58). The shape is the lesson: one neighbor copies noise, a thousand neighbors average away the signal, and the sweet spot sits in between. This is the class's first brush with the bias-variance trade-off, months before it gets that name. No need to formalize; just let them see the U.
>
> Suggested Day 1 stopping point.

---
## 3. Fixing Types and Formats
---

Our taxi file arrived in decent shape: numbers stored as numbers, only the datetimes needing repair. That is not the norm. Data typed by humans into spreadsheets arrives with numbers-as-text, inconsistent capitalization, stray spaces, and dates in whatever format the typist felt like that day.

To practice on something suitably crummy, here is a hand-entered log from a (fictional) TLC pilot program: five airport-shuttle trips recorded by an intern.

In [ ]:
# RUN-TOGETHER
shuttle_log = pd.DataFrame({
    'trip_id':  ['001', '002', '003', '004', '005'],
    'driver':   [' ALICE', 'bob ', 'Carla', ' BOB', 'alice'],
    'fare':     ['52.00', '47.50', '52.00', '61.25', '49.00'],
    'date':     ['03/04/2019', '03/04/2019', '03/05/2019', '03/06/2019', '03/07/2019'],
    'odometer': ['12,401', '12,438', '12,489', '12,532', '12,570'],
})

print(shuttle_log)
shuttle_log.info()

`info()` delivers the diagnosis: every column is stored as text, so every column is broken for its purpose. `fare` cannot be averaged, `date` cannot be sorted by time, `odometer` cannot be differenced, and `driver` contains what looks like five people but is probably two.

We will fix them one at a time. First, the easy numeric conversion, with `astype()`:

In [ ]:
# RUN-TOGETHER
shuttle_log['fare'] = shuttle_log['fare'].astype(float)

print(shuttle_log['fare'].mean().round(2))

`odometer` is harder: `astype(int)` would choke on the comma thousands-separators. Text repair comes first, and for that pandas gives every text column a `.str` accessor, the sibling of the `.dt` accessor from Module 7: `.str.strip()`, `.str.lower()`, `.str.replace()`, `.str.title()`, and friends, each applied to the whole column at once.

In [ ]:
# RUN-TOGETHER
# Remove the commas AS TEXT, then convert.
shuttle_log['odometer'] = shuttle_log['odometer'].str.replace(',', '').astype(int)

# Dates: give to_datetime the exact format rather than making it guess.
shuttle_log['date'] = pd.to_datetime(shuttle_log['date'], format='%m/%d/%Y')

shuttle_log.info()

Two conversions, one pattern: **repair the text, then convert the type.** The `format='%m/%d/%Y'` argument is worth the extra typing; date-format guessing is a classic source of silent errors (is 03/04/2019 March 4th or April 3rd? The answer depends on which side of the Atlantic typed it, so say what you mean).

What about `trip_id`? It *looks* numeric, and converting it with `astype(int)` would work without an error. We are going to leave it as text, on purpose, for two reasons: converting would destroy the leading zeros (`'001'` becomes `1`), and we will never do arithmetic on an ID (the "average trip ID" means nothing). **Identifiers are labels, not quantities.** Storable as numbers; better kept as text.

### Activity 4 (short code, about 5 minutes)

Finish the job: standardize the `driver` column so each name is stripped of stray spaces and capitalized like a name (`'Alice'`, `'Bob'`, `'Carla'`). Then count the trips per driver. How many actual drivers does the log contain?

In [ ]:
# Activity 4
shuttle_log['driver'] = shuttle_log['driver'].str.____().str.____()

print(shuttle_log)
print()
print(shuttle_log['driver'].____())

> **Instructor note (Activity 4).** `strip` then `title` (or `capitalize`). The payoff of standardizing: the five apparent drivers collapse to three (Alice 2, Bob 2, Carla 1). Point out the ordering moral: had we counted *before* cleaning, `' ALICE'` and `'alice'` would have been different people. Formatting problems are counting problems in disguise.

---
## 4. Duplicates
---

Duplicate rows are born whenever data changes hands: a file submitted twice, a merge run twice, a sensor that hiccups and resends. Each duplicate silently double-counts one observation in every statistic downstream.

Module 7 already checked our sample and found zero exact duplicates. To see the machinery work, we will vandalize a copy of our own data and then catch ourselves.

In [ ]:
# RUN-TOGETHER
# Append 3 randomly chosen rides a second time.
doubled = pd.concat([taxis, taxis.sample(3, random_state=8)])
print('shape after vandalism:', doubled.shape)

print('duplicated rows found:', doubled.duplicated().sum())

`duplicated()` marks a row True when an *identical row appeared earlier*; by default the first appearance stays False. So 3 planted copies, 3 Trues.

**Predict before running:** what will `duplicated(keep=False)` return as a sum? (`keep=False` marks *every* member of a duplicate family, originals included.)

In [ ]:
# RUN-TOGETHER
print('keep=False marks:', doubled.duplicated(keep=False).sum(), 'rows')

# And the repair, one line:
repaired = doubled.drop_duplicates()
print('after drop_duplicates:', repaired.shape)

Six, not three: each planted copy plus its original. And `drop_duplicates()` restores exactly 6,433 rows.

The judgment call hiding in this one-liner is the `subset` argument: **what counts as "the same ride"?** By default, all 17 columns must match. But suppose two rows agree on pickup time, dropoff time, and zones, yet differ by a cent of tip; are they two rides or one ride recorded twice? `duplicated(subset=[...])` lets you define identity yourself, and in messy real pipelines choosing that subset *is* the analysis. With second-resolution timestamps, full-row matches are almost certainly true duplicates; with coarser data, be more careful.

---
## 5. Combining Datasets
---

Outside the classroom, the data you need is rarely in one table. Ride records live in one file, zone information in another, driver records in a third, and the analysis you want requires stitching them together. The tool is `pd.merge()`, and it is a direct cousin of the JOIN operation that runs the database world.

The mechanics first, on a small example we can fully see; then a real payoff on the taxis.

In [ ]:
# RUN-TOGETHER
# A tiny product catalog, and a tiny sales ledger.
products = pd.DataFrame({
    'product_id': [101, 102, 103, 104],
    'name': ['Widget', 'Gadget', 'Doohickey', 'Gizmo'],
})
sales = pd.DataFrame({
    'product_id': [101, 102, 104, 105],
    'units_sold': [134, 243, 76, 100],
})
print(products)
print()
print(sales)

Note the imperfect overlap, which is the realistic part: product 103 never sold, and product 105 was sold but is missing from the catalog. Every merge must decide what happens to such orphans, and the `how` argument is that decision:

| `how=` | Keeps |
| --- | --- |
| `'inner'` | only keys present in **both** tables |
| `'left'` | **all** rows of the left table, matched where possible |
| `'right'` | all rows of the right table |
| `'outer'` | **everything**, filling gaps with NaN |

In [ ]:
# RUN-TOGETHER
print(pd.merge(products, sales, on='product_id', how='inner'))
print()
print(pd.merge(products, sales, on='product_id', how='left'))
print()
print(pd.merge(products, sales, on='product_id', how='outer'))

Read the three results against the table above: inner keeps 3 rows (the overlap), left keeps all 4 products with a NaN for unsold 103, outer keeps 5 rows including the mystery product 105 with no name. None of these is "the right join"; each answers a different question, and NaN entries in a merge result are often *findings* (an uncatalogued product!) rather than problems.

### 5.1 A real merge: are taxis serving the whole city?

Module 7 established that this sample is overwhelmingly Manhattan pickups. But Manhattan also has lots of *people*. Fair comparison requires rides **per resident**, and residential populations are not in our dataset. They are, however, publicly known: the 2020 U.S. Census counted the five boroughs. So: build a small table of facts from outside the data, and merge it on.

In [ ]:
# RUN-TOGETHER
# Borough populations, 2020 U.S. Census.
population = pd.DataFrame({
    'borough': ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island'],
    'residents': [1472654, 2736074, 1694251, 2405464, 495747],
})

# Ride counts per pickup borough, converted from a Series to a mergeable table.
ride_counts = taxis['pickup_borough'].value_counts().reset_index()
ride_counts.columns = ['borough', 'rides']
print(ride_counts)

In [ ]:
# RUN-TOGETHER
per_capita = pd.merge(ride_counts, population, on='borough', how='left')
per_capita['rides_per_100k'] = (per_capita['rides'] / per_capita['residents'] * 100_000).round(1)
per_capita

Now the Manhattan skew has a fair denominator, and it is even more extreme than the raw counts suggested: about **311 sampled rides per 100,000 Manhattan residents versus 7 per 100,000 in the Bronx**, a ratio of roughly 46 to 1. Street-hail taxi service, at least in this sample, is essentially a Manhattan phenomenon. That is a publishable-quality sentence, and it required a merge, because the crucial denominator lived outside our dataset.

One borough is missing from that result, though. Where did Staten Island go? Our left table was `ride_counts`, which has no Staten Island row (no sampled pickups there, recall Module 7 Activity 3), so `how='left'` never looked for it. Watch the join type change the story:

In [ ]:
# RUN-TOGETHER
pd.merge(ride_counts, population, on='borough', how='outer')

With `how='outer'`, Staten Island reappears, with `NaN` rides, and NaN here *is the finding*: a borough of nearly half a million people with zero street-hail pickups in the sample. The left join silently hid that fact; the outer join surfaced it. Join choice is not plumbing; it decides which absences you get to see.

### Activity 5 (short code, about 8 minutes)

Repeat the per-capita analysis for **dropoffs**:

a. Build `drop_counts` from `dropoff_borough` (mind the column renaming).

b. Merge with `population` using an **outer** join and compute `rides_per_100k`.

c. In a comment: which borough's number changed most interestingly compared to the pickup version, and what does it say about how taxis serve it?

In [ ]:
# Activity 5
drop_counts = taxis[____].value_counts().reset_index()
drop_counts.columns = ['borough', 'rides']

per_capita_drop = pd.merge(drop_counts, population, on=____, how=____)
per_capita_drop['rides_per_100k'] = (per_capita_drop['rides'] / per_capita_drop['residents'] * 100_000).round(1)
per_capita_drop

# c. your interpretation:
#

> **Instructor note (Activity 5).** Staten Island is the star: it has 2 dropoffs (0.4 per 100k) versus zero pickups; taxis will *take* you there but do not *start* there. Brooklyn and the Bronx also gain modestly as dropoff boroughs relative to pickups (people taxi home to the outer boroughs). Fine discussion of asymmetric service.
>
> Suggested Day 2 stopping point.

---
## 6. Outliers: Deciding, Not Just Detecting
---

Module 7 left outliers with a slogan: *a flag, not a verdict*. Today we deliver the verdict options. For flagged values that are not outright errors, the honest menu has four items:

1. **Keep, and document.** The default. Real long trips are part of reality.
2. **Remove, with the criterion written down.** Justified when values are errors, or when the analysis explicitly excludes a population ("local trips only, defined as under 10 miles").
3. **Transform.** Work on a scale where the skew behaves, most commonly logarithms.
4. **Separate into a labeled subpopulation.** When the "outliers" are really a different *kind* of observation wearing the same columns.

Options 3 and 4 deserve demonstrations, because both turn a nuisance into insight.

In [ ]:
# RUN-TOGETHER
# Option 3 preview: the fare distribution on a log scale.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=taxis, x='fare', ax=axes[0])
axes[0].set_title('fare, linear scale')
sns.histplot(data=taxis, x='fare', log_scale=True, ax=axes[1])
axes[1].set_title('fare, log scale')
plt.show()

On the log scale the crushed, long-tailed distribution opens up into something almost symmetric, and the flat-fare bump near 52 dollars stands out sharper than ever. Nothing was deleted; we just changed the ruler. Log transforms get a fuller treatment when we reach modeling; for now, know that "take logs of money-like variables" is one of the oldest tricks in applied statistics.

Option 4 is the flag-column strategy, and our dataset has been begging for it all module: airport rides *are* a subpopulation (long, expensive, sometimes flat-fare), so let us label them once and for all. The `isin()` method makes short work of it.

In [ ]:
# RUN-TOGETHER
airport_zones = ['JFK Airport', 'LaGuardia Airport']

taxis['is_airport'] = (taxis['pickup_zone'].isin(airport_zones) |
                       taxis['dropoff_zone'].isin(airport_zones))
taxis['is_flat_fare'] = taxis['fare'] == 52

print('airport rides:  ', taxis['is_airport'].sum(),
      '(', round(100 * taxis['is_airport'].mean(), 1), '% of sample )')
print('flat-fare rides:', taxis['is_flat_fare'].sum())

Flag columns cost nothing and delete nothing: every analysis downstream can now include, exclude, or compare airport rides with one boolean mask. This is what "cleaning" looks like at its best: not removing inconvenient data, but *adding structure* that makes the data honest to use.

And a flag pays for itself immediately. Module 7 left a conjecture hanging: that the strangely long 4-to-5 am rides were airport runs. We can now test it in one line.

In [ ]:
# RUN-TOGETHER
airport_share = taxis.groupby('pickup_hour')['is_airport'].mean().round(3)
print('airport share at 4 am:', airport_share[4])
print('airport share at 5 am:', airport_share[5])
print('airport share overall:', round(taxis['is_airport'].mean(), 3))

At 5 am, more than **one ride in five** touches an airport, versus about one in sixteen overall. The pre-dawn airport-run conjecture, born from a histogram in Module 7, is now supported by a variable that did not exist until we manufactured it. Notice the full loop: explore, notice, conjecture, engineer a column, confirm. That two-way arrow in the pipeline diagram is not decoration.

### Activity 6 (short code, about 5 minutes)

Module 7's IQR fence for distance was 6.55 miles. Build the flag `is_long = taxis['distance'] > 6.55`, then answer: **what fraction of long rides are airport rides?** Predict first.

In [ ]:
# Activity 6
taxis['is_long'] = taxis['distance'] > ____

print('long rides:', taxis['is_long'].sum())
print('airport share among long rides:',
      taxis.loc[taxis[____], 'is_airport'].____().round(2))

> **Instructor note (Activity 6).** 734 long rides, of which 49% are airport rides. So half of the "outliers" the IQR rule flagged in Module 7 are one identifiable subpopulation. The other half is a good open question for strong students (long non-airport rides: outer-borough crosstown trips, mostly). The punchline to say out loud: last module we would have deleted these; this module we understand them.

---
## 7. The Capstone: Building `taxis_clean`
---

Everything converges here. We now walk the Module 7 to-do list and make every call, out loud, with reasons. Our target analysis, agreed in advance per Rule 3: **a general-purpose table for studying fares and trips.** Decisions:

| # | Problem | Decision | Reason |
| --- | --- | --- | --- |
| 1 | 51 zero-distance rides with positive fares | **Drop** | For fare-and-trip analysis these are unusable: the key measurement failed. (The 6 impossible-duration rides all sit inside this group, so one criterion handles both.) |
| 2 | 96 rides with `passengers == 0` | **Fix to missing** | We judged these disguised missing values (Section 2.1); making them officially missing is honest. |
| 3 | 44 missing `payment` | **Fill with `'unknown'`** | An honest label; the mode would stamp "credit card" on cash-like rides (Section 2.4). |
| 4 | Missing zones and boroughs | **Keep as-is** | Not needed for most fare analyses; dropping whole rides for them wastes good data. |
| 5 | Airport and flat-fare subpopulations | **Flag columns** | Section 6's strategy: add structure, delete nothing. |
| 6 | Cash tips recorded as zero | **Cannot be fixed; document** | No code can recover unrecorded tips. The documentation *is* the fix. |

One quiet consequence of decision 2, promised back in Section 2: the moment `passengers` gains missing values it will convert from integer to float. Watch for it.

In [ ]:
# RUN-TOGETHER
# The cleaning script. Every block: one decision, one reason.

# Start from the repaired raw table (never the raw file itself: Rule 1).
taxis_clean = taxis.copy()

# Decision 1: drop zero-distance rides with positive fares
# (unusable for trip analysis; includes all impossible durations).
broken = (taxis_clean['distance'] == 0) & (taxis_clean['fare'] > 0)
print('dropping', broken.sum(), 'rows with failed distance measurement')
taxis_clean = taxis_clean[~broken]

# Decision 2: passengers == 0 is a disguised missing value.
taxis_clean['passengers'] = taxis_clean['passengers'].replace(0, np.nan)

# Decision 3: missing payment becomes an honest 'unknown' category.
taxis_clean['payment'] = taxis_clean['payment'].fillna('unknown')

# Decision 4: zone/borough gaps stay; they are documented, not hidden.
# Decision 5: flag columns is_airport / is_flat_fare already built in Section 6.
# Decision 6: no code. See the data notes below.

print('final shape:', taxis_clean.shape)

In [ ]:
# RUN-TOGETHER
# Verify the result the same way we would inspect any new dataset.
print(taxis_clean.isna().sum()[taxis_clean.isna().sum() > 0])
print()
print('passengers dtype now:', taxis_clean['passengers'].dtype)
print('payment categories:  ', dict(taxis_clean['payment'].value_counts()))
print('flat-fare rides kept:', taxis_clean['is_flat_fare'].sum())

Read the verification like an auditor:

- 6,382 rows remain (51 dropped, each for a written reason).
- `passengers` now shows 95 missing (one of the 96 zero-passenger rides was itself a zero-distance ride, already dropped) and its dtype quietly became `float64`, exactly as predicted in Section 2. Integer plus NaN equals float; now you have seen it live.
- `payment` has three honest categories: 4,554 credit card, 1,789 cash, 39 unknown.
- 126 flat-fare rides survive; 5 of the original 131 were dropped, because they were zero-distance rides. Even the JFK flat fare is not immune to broken meters.

Last step of any cleaning job: save the product, and write the label that goes on the jar.

In [ ]:
# RUN-TOGETHER
taxis_clean.to_csv('taxis_clean.csv', index=False)

data_notes = '''
TAXIS_CLEAN, v1 (built from the seaborn taxis sample, 6433 rides, March 2019)
- 6382 rows. Dropped 51 rides with distance == 0 and fare > 0 (failed measurement;
  includes all 6 rides with non-positive duration).
- passengers: 0 replaced by missing (95 rides); treated as a recording default.
- payment: 44 missing values labeled 'unknown'; these rides record zero tip and
  resemble cash rides.
- tip: card tips only. Cash tips are NOT recorded (TLC meter limitation).
  Do not compare tips across payment types or cab colors without addressing this.
- is_airport, is_flat_fare, is_long: analyst-added flags, definitions in Module 8.
- Zone/borough columns retain 15 to 30 missing values each.
'''
print(data_notes)

Those dozen lines of notes may be the most valuable output of the entire module. A stranger who receives `taxis_clean.csv` *with* the notes can use it safely; a stranger who receives it without them will, sooner or later, publish a comparison of tips by payment type. Data plus documentation is a dataset; data alone is a trap with good intentions.

### Activity 7 (capstone: clean it differently, 20-30 minutes, pairs)

Rule 3 says cleaning depends on purpose, and now you will prove it. Your assignment: build **`taxis_tips`**, a table prepared for one specific study: *"How do New Yorkers tip on taxi rides?"*

Work from `taxis_clean` and make your own decisions. The parts below scaffold the reasoning; the final decisions are yours to make and defend.

In [ ]:
# Activity 7, Part A: which rides can participate in a tipping study AT ALL?
# (Think about Section 2 and the data notes before filling this in.)
taxis_tips = taxis_clean[taxis_clean['payment'] == ____].copy()
print('rides in the tipping table:', taxis_tips.shape[0])

In [ ]:
# Activity 7, Part B: build the variable a tipping study actually wants:
# tip as a PERCENTAGE of fare. Watch out: what could go wrong in the division?
taxis_tips['tip_pct'] = (100 * taxis_tips[____] / taxis_tips[____]).round(1)
print(taxis_tips['tip_pct'].describe().round(1))

In [ ]:
# Activity 7, Part C: quality sweep on YOUR new variable, your way.
# At minimum: a histogram of tip_pct, and a look at its most extreme values.
# (No blanks. You have all the tools.)

**Part D (the memo).** In a markdown cell, document `taxis_tips` the way we documented `taxis_clean`:

- which rows you kept and why the excluded ones could not participate;
- one decision where a reasonable classmate might have chosen differently (for instance: what did you do with the 39 `'unknown'` payments? With zero-tip card rides?);
- one caution inherited from the raw data that no amount of cleaning removed;
- one question about tipping your table is now ready to answer.

> **Instructor note (Activity 7).** Part A: the defensible core answer is credit card rides only (4,554 rows), since cash tips are unrecorded; keeping 'unknown' rides is defensible only with an argument, and that argument appearing in a memo is a win. Part B: `tip_pct`; division by zero is impossible here only because minimum fare is positive, worth asking who checked. Part C findings: median tip percentage 25.6 of the metered fare (a nice discussion point: card readers suggest tips as a percentage of the *total* including surcharges and tax, so tip-over-fare runs higher than the felt 20 percent), a spike at exactly 0 (443 card rides, about 10%, tip nothing: real non-tippers this time, not an artifact!), and extreme percentages up to 93 coming from small fares. That 0-spike contrast with Module 7's artifact spike is the deepest callback in the module; make sure it lands.
>
> Common trap in Part D: forgetting that even this table says nothing about cash tippers. The inherited caution never goes away.

### 7.1 Closing reflection (exit ticket)

1. In one sentence each, defend two decisions from the `taxis_clean` script to a skeptical boss who says "you deleted my data".
2. Name one decision in this module where the *analysis goal* flipped the right answer.
3. We graded imputation methods with hidden-value experiments. Why can't we grade our real cleaning decisions the same way, and what do we rely on instead?

---
## References and further reading

- Huber, F. *Hands-on Introduction to Data Science with Python*, chapter on data cleaning and preprocessing.
- pandas user guide: [Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) and [Merge, join, concatenate](https://pandas.pydata.org/docs/user_guide/merging.html).
- scikit-learn user guide: [Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html).
- NYC Taxi and Limousine Commission, [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) (data dictionaries, including the cash-tip limitation).
- Borough populations: 2020 U.S. Census, via [census.gov](https://www.census.gov/).

*Where we go next: with `taxis_clean` in hand and its documentation attached, we can finally trust our summaries and pictures enough to build on them. The EXPLORE box is behind us; MODEL is ahead.*